In [12]:
## RUN this BEFORE job submission

import os

_new_directory = "/home/pkwon2/aragonite/mpnn_post_af3" ## change to correct directory

print(f"The current working directory is: {os.getcwd()}")

os.chdir(_new_directory)
print(f"Changed working directory to: {os.getcwd()}")

def create_directory_structure(base_directory):
    try:
        os.makedirs(os.path.join(base_directory, 'input'), exist_ok=True)
        os.makedirs(os.path.join(base_directory, 'outputs'), exist_ok=True)
        print(f"Directory structure '{base_directory}/input' and '{base_directory}/output' created successfully.")
    except Exception as e:
        print(f"Error creating directory structure '{base_directory}': {e}")

if __name__ == "__main__":
    base_directory = "alphafold3"
    create_directory_structure(base_directory)

The current working directory is: /home/pkwon2/aragonite/mpnn_post_af3
Changed working directory to: /home/pkwon2/aragonite/mpnn_post_af3
Directory structure 'alphafold3/input' and 'alphafold3/output' created successfully.


In [6]:
import os
from Bio.PDB import PDBParser, PPBuilder

# Define input and output directories
pdb_dir = "/home/pkwon2/aragonite/mpnn_post_af3/outputs/packed"
fasta_dir = "/home/pkwon2/aragonite/mpnn_post_af3/outputs/fasta"

# Ensure output directory exists
os.makedirs(fasta_dir, exist_ok=True)

# Initialize PDB parser and peptide builder
parser = PDBParser(QUIET=True)
ppb = PPBuilder()

# Process each PDB file
for pdb_file in os.listdir(pdb_dir):
    if pdb_file.endswith(".pdb"):
        pdb_path = os.path.join(pdb_dir, pdb_file)
        structure_id = os.path.splitext(pdb_file)[0]
        structure = parser.get_structure(structure_id, pdb_path)

        first_chain_seq = None
        for model in structure:
            for chain in model:
                peptides = ppb.build_peptides(chain)
                chain_seq = "".join(str(pp.get_sequence()) for pp in peptides)
                if chain_seq:
                    first_chain_seq = chain_seq.strip()
                    break  # stop after first chain
            break  # only first model

        if first_chain_seq:
            # Repeat sequence 6 times separated by spaces
            repeated_sequence = " ".join([first_chain_seq] * 6)

            fasta_path = os.path.join(fasta_dir, f"{structure_id}.fasta")
            with open(fasta_path, "w") as f:
                f.write(f">{structure_id}\n{repeated_sequence}\n")

# import os
# from Bio.PDB import PDBParser, PPBuilder

# # Define input and output directories
# pdb_dir = "/home/pkwon2/aragonite/mpnn_post_af3/outputs/packed"
# fasta_dir = "/home/pkwon2/aragonite/mpnn_post_af3/outputs/fasta"

# # Ensure output directory exists
# os.makedirs(fasta_dir, exist_ok=True)

# # Initialize PDB parser and peptide builder
# parser = PDBParser(QUIET=True)
# ppb = PPBuilder()

# # Process each PDB file
# for pdb_file in os.listdir(pdb_dir):
#     if pdb_file.endswith(".pdb"):
#         pdb_path = os.path.join(pdb_dir, pdb_file)
#         structure_id = os.path.splitext(pdb_file)[0]
#         structure = parser.get_structure(structure_id, pdb_path)

#         chain_sequences = []
#         for model in structure:
#             for chain in model:
#                 if chain.id not in ["A", "B", "C"]:
#                     continue  # Skip chains not A, B, or C

#                 peptides = ppb.build_peptides(chain)
#                 chain_seq = "".join(str(pp.get_sequence()) for pp in peptides)
#                 if chain_seq:
#                     chain_sequences.append(chain_seq.strip())

#             break  # Only process the first model

#         if chain_sequences:
#             combined_sequence = " ".join(chain_sequences).strip()
#             fasta_path = os.path.join(fasta_dir, f"{structure_id}.fasta")
#             with open(fasta_path, "w") as f:
#                 f.write(f">{structure_id}\n{combined_sequence}\n")

In [7]:
base_directory = "alphafold3"
create_directory_structure(base_directory)

import os
import glob

def main():
    # Define the absolute path for the tasks file
    tasks_file = "/home/pkwon2/aragonite/mpnn_post_af3/alphafold3/tasks"
    
    # Correct directory containing FASTA files
    fasta_dir = "/home/pkwon2/aragonite/mpnn_post_af3/outputs/fasta"
    
    # Find all FASTA files in the fasta_dir
    fasta_files = glob.glob(os.path.join(fasta_dir, "*.fasta"))
    
    # Write one command per FASTA file into the tasks file
    with open(tasks_file, "w") as file:
        for fasta_file in fasta_files:
            command = (
                "/software/containers/users/ikalvet/mlfold3/mlfold3_01.sif "
                "/net/software/lab/alphafold3/run_alphafold_custom.py "
                f"--fasta {fasta_file} "
                "--run_data_pipeline=false "
                "--output_dir ./outputs\n"
            )
            file.write(command)
    
    print(f"Tasks file '{tasks_file}' has been created with {len(fasta_files)} commands.")

if __name__ == "__main__":
    main()

Directory structure 'alphafold3/input' and 'alphafold3/output' created successfully.
Tasks file '/home/pkwon2/aragonite/mpnn_post_af3/alphafold3/tasks' has been created with 100 commands.


In [8]:
#SBATCH -p gpu
#SBATCH --mem=48g
#SBATCH -t 00:30:00
#SBATCH -c 12
#SBATCH --gres=gpu:a6000:1
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J af3
#SBATCH -p gpu-train
#SBATCH --mem=48g
#SBATCH -t 01:00:00
#SBATCH -c 8
#SBATCH --gres=gpu:a100:2
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

SBATCH script 'alphafold3/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/pkwon2/aragonite/mpnn_post_af3/alphafold3
sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh


In [23]:
#!/usr/bin/env python3
import os, json, shutil
from glob import glob

# Paths
af_dirs = glob("/home/pkwon2/aragonite/mpnn_post_af3/alphafold3/outputs/*")
out_dir = "/home/pkwon2/aragonite/mpnn_post_af3/alphafold3/filtered"
os.makedirs(out_dir, exist_ok=True)

# Thresholds
iptm_min = 0.4
ptm_min  = 0.47

# Scan AF outputs
for d in af_dirs:
    for cif_fp in glob(os.path.join(d, "*_model.cif")):
        base = cif_fp[:-len("_model.cif")]
        json_fp = f"{base}_summary_confidences.json"

        if not os.path.exists(json_fp):
            print(f"[SKIP] No JSON for {base}")
            continue

        try:
            with open(json_fp) as f:
                conf = json.load(f)
            iptm = float(conf.get("iptm") or 0.0)
            ptm  = float(conf.get("ptm") or 0.0)
        except Exception as e:
            print(f"[ERR] {base}: {e}")
            continue

        if iptm > iptm_min and ptm > ptm_min:
            shutil.copy2(cif_fp, out_dir)
            shutil.copy2(json_fp, out_dir)
            print(f"[KEEP] {os.path.basename(base)} (ipTM={iptm:.3f}, pTM={ptm:.3f})")
        else:
            print(f"[DROP] {os.path.basename(base)} (ipTM={iptm:.3f}, pTM={ptm:.3f})")

[DROP] d12_for_mpnn_packed_68_1 (ipTM=0.400, pTM=0.480)
[KEEP] d12_for_mpnn_packed_30_1 (ipTM=0.540, pTM=0.600)
[KEEP] d12_for_mpnn_packed_11_1 (ipTM=0.510, pTM=0.580)
[KEEP] d12_for_mpnn_packed_49_1 (ipTM=0.560, pTM=0.620)
[DROP] d12_for_mpnn_packed_85_1 (ipTM=0.290, pTM=0.370)
[KEEP] d12_for_mpnn_packed_72_1 (ipTM=0.480, pTM=0.540)
[DROP] d12_for_mpnn_packed_53_1 (ipTM=0.290, pTM=0.390)
[KEEP] d12_for_mpnn_packed_1_1 (ipTM=0.430, pTM=0.510)
[KEEP] d12_for_mpnn_packed_43_1 (ipTM=0.540, pTM=0.600)
[KEEP] d12_for_mpnn_packed_95_1 (ipTM=0.470, pTM=0.540)
[DROP] d12_for_mpnn_packed_62_1 (ipTM=0.330, pTM=0.420)
[KEEP] d12_for_mpnn_packed_59_1 (ipTM=0.530, pTM=0.590)
[DROP] d12_for_mpnn_packed_20_1 (ipTM=0.300, pTM=0.390)
[KEEP] d12_for_mpnn_packed_78_1 (ipTM=0.510, pTM=0.570)
[KEEP] d12_for_mpnn_packed_21_1 (ipTM=0.510, pTM=0.570)
[KEEP] d12_for_mpnn_packed_79_1 (ipTM=0.430, pTM=0.500)
[DROP] d12_for_mpnn_packed_58_1 (ipTM=0.390, pTM=0.470)
[DROP] d12_for_mpnn_packed_63_1 (ipTM=0.370, pTM=

In [24]:
#!/usr/bin/env python3
import os
import csv
import numpy as np
from glob import glob
from Bio.PDB.MMCIFParser import MMCIFParser

# Paths
in_dir  = "/home/pkwon2/aragonite/mpnn_post_af3/alphafold3/filtered"
out_csv = os.path.join(in_dir, "flatness_metrics.csv")

# Parser
parser = MMCIFParser(QUIET=True)

def get_calpha_coords(cif_file):
    """Extract C-alpha coordinates from an mmCIF file."""
    structure = parser.get_structure("struct", cif_file)
    coords = []
    for model in structure:
        for chain in model:
            for residue in chain:
                try:
                    atom = residue["CA"]  # dictionary-style access
                    coords.append(atom.coord)
                except KeyError:
                    continue
    return np.array(coords)

def compute_flatness(coords):
    """
    Compute flatness as variance of absolute distances from best-fit plane.
    Lower values = flatter.
    """
    if len(coords) < 3:
        return None
    # Center
    coords_centered = coords - coords.mean(axis=0)
    # PCA via SVD
    _, _, vh = np.linalg.svd(coords_centered)
    normal = vh[-1]  # normal to best-fit plane
    # Distances from plane
    dists = np.abs(coords_centered.dot(normal))
    # Use variance of absolute distances
    return np.var(dists)

def main():
    cif_files = glob(os.path.join(in_dir, "*_model.cif"))
    results = []

    for cif_fp in cif_files:
        coords = get_calpha_coords(cif_fp)
        flatness = compute_flatness(coords)
        base = os.path.basename(cif_fp)
        if flatness is not None:
            results.append((base, flatness))
            print(f"[OK] {base}: flatness_var = {flatness:.6f}")
        else:
            print(f"[SKIP] {base}: too few atoms")

    # Sort results by flatness (ascending = flattest first)
    results.sort(key=lambda x: x[1])

    # Write CSV
    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "flatness_variance"])
        writer.writerows(results)

    print(f"[DONE] Results written to {out_csv}")

if __name__ == "__main__":
    main()

[OK] d12_for_mpnn_packed_72_1_model.cif: flatness_var = 40.498814
[OK] d12_for_mpnn_packed_30_1_model.cif: flatness_var = 40.237865
[OK] d12_for_mpnn_packed_9_1_model.cif: flatness_var = 40.341156
[OK] d12_for_mpnn_packed_86_1_model.cif: flatness_var = 39.300533
[OK] d12_for_mpnn_packed_6_1_model.cif: flatness_var = 39.770855
[OK] d12_for_mpnn_packed_96_1_model.cif: flatness_var = 40.139038
[OK] d12_for_mpnn_packed_42_1_model.cif: flatness_var = 39.225990
[OK] d12_for_mpnn_packed_12_1_model.cif: flatness_var = 39.021229
[OK] d12_for_mpnn_packed_60_1_model.cif: flatness_var = 39.060940
[OK] d12_for_mpnn_packed_22_1_model.cif: flatness_var = 38.986042
[OK] d12_for_mpnn_packed_94_1_model.cif: flatness_var = 39.918819
[OK] d12_for_mpnn_packed_40_1_model.cif: flatness_var = 58.836876
[OK] d12_for_mpnn_packed_50_1_model.cif: flatness_var = 39.197327
[OK] d12_for_mpnn_packed_32_1_model.cif: flatness_var = 39.350502
[OK] d12_for_mpnn_packed_4_1_model.cif: flatness_var = 40.036667
[OK] d12_for_

In [25]:
#!/usr/bin/env python3
import os
import csv

# Paths
filtered_dir = "/home/pkwon2/aragonite/mpnn_post_af3/alphafold3/filtered"
metrics_csv  = os.path.join(filtered_dir, "flatness_metrics.csv")

# Threshold
flatness_cutoff = 40.5

def main():
    to_delete = []
    
    # Load metrics
    with open(metrics_csv) as f:
        reader = csv.DictReader(f)
        for row in reader:
            flatness = float(row["flatness_variance"])
            if flatness > flatness_cutoff:
                cif_file = os.path.join(filtered_dir, row["filename"])
                json_file = cif_file.replace(
                    "_model.cif",
                    "_summary_confidences.json"
                )
                to_delete.append((cif_file, json_file))
    
    # Delete files
    for cif_fp, json_fp in to_delete:
        if os.path.exists(cif_fp):
            os.remove(cif_fp)
            print(f"[DEL] {os.path.basename(cif_fp)}")
        if os.path.exists(json_fp):
            os.remove(json_fp)
            print(f"[DEL] {os.path.basename(json_fp)}")
    
    print(f"[DONE] Deleted {len(to_delete)} CIF+JSON pairs with flatness > {flatness_cutoff}")

if __name__ == "__main__":
    main()

[DEL] d12_for_mpnn_packed_1_1_model.cif
[DEL] d12_for_mpnn_packed_1_1_summary_confidences.json
[DEL] d12_for_mpnn_packed_97_1_model.cif
[DEL] d12_for_mpnn_packed_97_1_summary_confidences.json
[DEL] d12_for_mpnn_packed_35_1_model.cif
[DEL] d12_for_mpnn_packed_35_1_summary_confidences.json
[DEL] d12_for_mpnn_packed_11_1_model.cif
[DEL] d12_for_mpnn_packed_11_1_summary_confidences.json
[DEL] d12_for_mpnn_packed_71_1_model.cif
[DEL] d12_for_mpnn_packed_71_1_summary_confidences.json
[DEL] d12_for_mpnn_packed_48_1_model.cif
[DEL] d12_for_mpnn_packed_48_1_summary_confidences.json
[DEL] d12_for_mpnn_packed_28_1_model.cif
[DEL] d12_for_mpnn_packed_28_1_summary_confidences.json
[DEL] d12_for_mpnn_packed_25_1_model.cif
[DEL] d12_for_mpnn_packed_25_1_summary_confidences.json
[DEL] d12_for_mpnn_packed_73_1_model.cif
[DEL] d12_for_mpnn_packed_73_1_summary_confidences.json
[DEL] d12_for_mpnn_packed_21_1_model.cif
[DEL] d12_for_mpnn_packed_21_1_summary_confidences.json
[DEL] d12_for_mpnn_packed_69_1_m

In [26]:
#!/usr/bin/env python3
import os
from glob import glob

IN_DIR  = "/home/pkwon2/aragonite/mpnn_post_af3/alphafold3/filtered/cowboy/cif"
OUT_DIR = "/home/pkwon2/aragonite/mpnn_post_af3/alphafold3/filtered/cowboy/pdb"
os.makedirs(OUT_DIR, exist_ok=True)

def convert_with_gemmi(cif_path, pdb_path):
    import gemmi
    st = gemmi.read_structure(cif_path)
    # If multiple models exist, write them all into one PDB (MODEL records)
    with open(pdb_path, "wt") as fh:
        for i, mdl in enumerate(st):
            if len(st) > 1:
                fh.write(f"MODEL     {i+1}\n")
            w = gemmi.PdbWriter(fh)
            w.write_model(mdl)
            if len(st) > 1:
                fh.write("ENDMDL\n")
        fh.write("END\n")

def convert_with_biopython(cif_path, pdb_path):
    # Parse mmCIF -> Structure -> write PDB
    from Bio.PDB import MMCIFParser, PDBIO
    parser = MMCIFParser(QUIET=True)
    io = PDBIO()
    structure_id = os.path.splitext(os.path.basename(cif_path))[0]
    structure = parser.get_structure(structure_id, cif_path)
    io.set_structure(structure)
    io.save(pdb_path)

def main():
    cif_files = sorted(glob(os.path.join(IN_DIR, "*.cif")))
    if not cif_files:
        print(f"[WARN] No .cif files found in: {IN_DIR}")
        return

    # Check which backend is available
    use_gemmi = False
    try:
        import gemmi  # noqa: F401
        use_gemmi = True
        print("[INFO] Using gemmi for conversion.")
    except Exception:
        print("[INFO] gemmi not found; falling back to Biopython.")

    converted = 0
    for cif in cif_files:
        base = os.path.splitext(os.path.basename(cif))[0]
        pdb_out = os.path.join(OUT_DIR, f"{base}.pdb")

        try:
            if use_gemmi:
                convert_with_gemmi(cif, pdb_out)
            else:
                convert_with_biopython(cif, pdb_out)
            converted += 1
            print(f"[OK] {base}.cif -> {base}.pdb")
        except Exception as e:
            print(f"[ERR] Failed to convert {cif}: {e}")

    print(f"\n[DONE] Converted {converted}/{len(cif_files)} CIFs to PDBs in {OUT_DIR}")

if __name__ == "__main__":
    main()

[INFO] gemmi not found; falling back to Biopython.
[OK] d12_for_mpnn_packed_12_1_model.cif -> d12_for_mpnn_packed_12_1_model.pdb
[OK] d12_for_mpnn_packed_13_1_model.cif -> d12_for_mpnn_packed_13_1_model.pdb
[OK] d12_for_mpnn_packed_14_1_model.cif -> d12_for_mpnn_packed_14_1_model.pdb
[OK] d12_for_mpnn_packed_15_1_model.cif -> d12_for_mpnn_packed_15_1_model.pdb
[OK] d12_for_mpnn_packed_16_1_model.cif -> d12_for_mpnn_packed_16_1_model.pdb
[OK] d12_for_mpnn_packed_18_1_model.cif -> d12_for_mpnn_packed_18_1_model.pdb
[OK] d12_for_mpnn_packed_19_1_model.cif -> d12_for_mpnn_packed_19_1_model.pdb
[OK] d12_for_mpnn_packed_22_1_model.cif -> d12_for_mpnn_packed_22_1_model.pdb
[OK] d12_for_mpnn_packed_23_1_model.cif -> d12_for_mpnn_packed_23_1_model.pdb
[OK] d12_for_mpnn_packed_26_1_model.cif -> d12_for_mpnn_packed_26_1_model.pdb
[OK] d12_for_mpnn_packed_27_1_model.cif -> d12_for_mpnn_packed_27_1_model.pdb
[OK] d12_for_mpnn_packed_29_1_model.cif -> d12_for_mpnn_packed_29_1_model.pdb
[OK] d12_for_

In [ ]:
/net/software/lab/johnbercow/JohnBercow.py --verbose --order_pdbs ./ --order_name 2025_10_2_alt_array --design_prefix LM --design_id 4 --gg_vector LM0627 --species e_coli --no_layout